In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

In [2]:
data = pd.read_csv(
    r'C:\Users\Vineet\OneDrive\Desktop\credit_card_resume\data\creditcard.csv'
)

X = data.drop(columns=["Class"])
y = data["Class"]

print(X.shape)
print(y.shape)


(284807, 30)
(284807,)


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)


Training set: (227845, 30)
Test set: (56962, 30)


In [4]:
scaler = StandardScaler()

In [5]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [6]:
results = []

# Step 4: Cross-validation loop
for fold, (train_idx, val_idx) in enumerate(
    cv.split(X_train, y_train), start=1
):
    print(f"\nFold {fold}")

    # Split fold data
    X_fold_train = X_train.iloc[train_idx].copy()
    X_fold_val = X_train.iloc[val_idx].copy()

    y_fold_train = y_train.iloc[train_idx].to_numpy(
        dtype=np.float32
    )
    y_fold_val = y_train.iloc[val_idx].to_numpy(
        dtype=np.float32
    )

    # Step 5: Scale Time and Amount
    scaler = StandardScaler()

    X_fold_train[["Time", "Amount"]] = scaler.fit_transform(
        X_fold_train[["Time", "Amount"]]
    )

    X_fold_val[["Time", "Amount"]] = scaler.transform(
        X_fold_val[["Time", "Amount"]]
    )

    # Step 6: Convert to PyTorch tensors
    X_tr = torch.tensor(
        X_fold_train.to_numpy(dtype=np.float32)
    )
    y_tr = torch.tensor(y_fold_train)

    X_val = torch.tensor(
        X_fold_val.to_numpy(dtype=np.float32)
    )

    # Step 7: Create DataLoader
    train_dataset = TensorDataset(X_tr, y_tr)

    generator = torch.Generator().manual_seed(42 + fold)

    train_loader = DataLoader(
        train_dataset,
        batch_size=1000,
        shuffle=True,
        generator=generator
    )

    # Step 8: Initialize a new neural network
    torch.manual_seed(42 + fold)

    model = nn.Sequential(
        nn.Linear(X_tr.shape[1], 10),
        nn.ReLU(),
        nn.Linear(10, 1)
    )

    # Step 9: Loss function and optimizer
    criterion = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.001
    )

    # Step 10: Train the model
    for epoch in range(50):
        model.train()

        for X_batch, y_batch in train_loader:
            logits = model(X_batch).flatten()
            loss = criterion(logits, y_batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # Step 11: Validation predictions
    model.eval()

    with torch.no_grad():
        logits = model(X_val).flatten()
        probs = torch.sigmoid(logits).numpy()

    predictions = (probs >= 0.5).astype(int)
    y_true = y_fold_val.astype(int)

    # Step 12: Calculate metrics
    fold_results = {
        "fold": fold,
        "accuracy": accuracy_score(y_true, predictions),
        "precision": precision_score(
            y_true, predictions, zero_division=0
        ),
        "recall": recall_score(
            y_true, predictions, zero_division=0
        ),
        "f1": f1_score(
            y_true, predictions, zero_division=0
        ),
        "roc_auc": roc_auc_score(y_true, probs),
        "pr_auc": average_precision_score(y_true, probs)
    }

    results.append(fold_results)
    print(fold_results)


Fold 1
{'fold': 1, 'accuracy': 0.9994952709078541, 'precision': 0.9661016949152542, 'recall': 0.7307692307692307, 'f1': 0.8321167883211679, 'roc_auc': 0.9865529896305215, 'pr_auc': 0.8693769077037754}

Fold 2
{'fold': 2, 'accuracy': 0.9994733261647173, 'precision': 0.9104477611940298, 'recall': 0.7721518987341772, 'f1': 0.8356164383561644, 'roc_auc': 0.9701125021217627, 'pr_auc': 0.8221640159011734}

Fold 3
{'fold': 3, 'accuracy': 0.9993416577058966, 'precision': 0.8450704225352113, 'recall': 0.759493670886076, 'f1': 0.8, 'roc_auc': 0.9701483981734754, 'pr_auc': 0.7824645386705902}

Fold 4
{'fold': 4, 'accuracy': 0.9994513814215804, 'precision': 0.8461538461538461, 'recall': 0.8354430379746836, 'f1': 0.8407643312101911, 'roc_auc': 0.9822726374693562, 'pr_auc': 0.8701943811910965}

Fold 5
{'fold': 5, 'accuracy': 0.9994513814215804, 'precision': 0.8970588235294118, 'recall': 0.7721518987341772, 'f1': 0.8299319727891157, 'roc_auc': 0.9866461122349883, 'pr_auc': 0.8270899485400183}


In [7]:
# Step 13: Summarize CV results
results_df = pd.DataFrame(results)

print("\n--- Results by Fold ---")
print(results_df.round(4).to_string(index=False))

print("\n--- Mean CV Scores ---")
print(
    results_df.drop(columns="fold").mean().round(4)
)

print("\n--- Standard Deviation ---")
print(
    results_df.drop(columns="fold").std().round(4)
)


--- Results by Fold ---
 fold  accuracy  precision  recall     f1  roc_auc  pr_auc
    1    0.9995     0.9661  0.7308 0.8321   0.9866  0.8694
    2    0.9995     0.9104  0.7722 0.8356   0.9701  0.8222
    3    0.9993     0.8451  0.7595 0.8000   0.9701  0.7825
    4    0.9995     0.8462  0.8354 0.8408   0.9823  0.8702
    5    0.9995     0.8971  0.7722 0.8299   0.9866  0.8271

--- Mean CV Scores ---
accuracy     0.9994
precision    0.8930
recall       0.7740
f1           0.8277
roc_auc      0.9791
pr_auc       0.8343
dtype: float64

--- Standard Deviation ---
accuracy     0.0001
precision    0.0504
recall       0.0383
f1           0.0160
roc_auc      0.0084
pr_auc       0.0368
dtype: float64
